# Practical 1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv('customer_data.csv')
df.head()

In [ ]:
# Histogram
df.hist(bins=30, figsize=(14, 8))
plt.show()

In [ ]:
# Numeric columns, correlation matrix, heatmap
num_df = df.select_dtypes(include='number')
corr = num_df.corr()
print(corr)
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

# Practical 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
df = pd.read_csv('customer_data.csv')
print(df.shape)

In [ ]:
# Drop duplicates, fix "XXXX", parse date
df = df.drop_duplicates()
print(df.shape)
df['Customer_Lifespan_Months'] = df['Customer_Lifespan_Months'].replace('XXXX', np.nan).astype(float)
date_col = 'Date'    # <-- change to the real date column name
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
df.info()

In [ ]:
# Fill missing values
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(exclude=['number', 'datetime']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])
df[date_col] = df[date_col].ffill()
print(df.isnull().sum())

In [ ]:
# Label encoding
le = LabelEncoder()
for c in cat_cols:
    df[c] = le.fit_transform(df[c])
df.head()

In [ ]:
# Outliers using box plot
plt.figure(figsize=(14, 6))
for i, c in enumerate(num_cols):
    plt.subplot(1, len(num_cols), i + 1)
    sns.boxplot(y=df[c])
    plt.title(c)
plt.tight_layout()
plt.show()

# Practical 3

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv('penguins.csv')
col = 'species'     # <-- check the actual column name
print(df[col].value_counts(normalize=True))

In [ ]:
# Simple random sampling (20%)
simple = df.sample(frac=0.2, random_state=1)
# Stratified sampling (20%, proportions kept)
_, strat = train_test_split(df, test_size=0.2, stratify=df[col], random_state=1)

In [ ]:
# Compare proportions
comparison = pd.DataFrame({
    'Population': df[col].value_counts(normalize=True),
    'Simple': simple[col].value_counts(normalize=True),
    'Stratified': strat[col].value_counts(normalize=True)
})
print(comparison.round(3))

# Practical 4

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
num_samples = 10000
sample_size = 30
population = np.random.exponential(scale=2.0, size=num_samples * sample_size)
sample_means = []
for i in range(num_samples):
    sample = np.random.exponential(scale=2.0, size=sample_size)
    sample_means.append(np.mean(sample))

In [ ]:
plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
sns.histplot(population, bins=50, kde=True, color="skyblue")
plt.title("Original Exponential Distribution")
plt.subplot(1, 2, 2)
sns.histplot(sample_means, bins=50, kde=True, color="salmon")
plt.title(f"Distribution of Sample Means (n={sample_size})")
plt.show()
print("Population mean:", np.mean(population))
print("Mean of sample means:", np.mean(sample_means))
print("Std of sample means:", np.std(sample_means))

# Practical 5

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv('iris.csv')
col = 'SepalLengthCm'    # <-- or 'sepal_length'; check df.columns
data = df[col]
print("Original mean:", data.mean())

In [ ]:
# Bootstrap
boot_means = []
for i in range(1000):
    sample = np.random.choice(data, size=len(data), replace=True)
    boot_means.append(np.mean(sample))
print("Bootstrap mean:", np.mean(boot_means))
print("Standard error:", np.std(boot_means))
print("95% CI:", np.percentile(boot_means, [2.5, 97.5]))

In [ ]:
# Plot
plt.figure(figsize=(8, 5))
sns.histplot(boot_means, bins=30, kde=True, color="salmon")
plt.axvline(np.percentile(boot_means, 2.5), color='black', linestyle='--')
plt.axvline(np.percentile(boot_means, 97.5), color='black', linestyle='--')
plt.title("Bootstrap Distribution of Sample Mean")
plt.show()

# Practical 6

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
df = pd.read_csv('iris.csv')
features = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']   # check names

In [ ]:
# MLE of mean and std for each feature (normal assumption)
for c in features:
    x = df[c]
    print(c, "-> mu =", round(x.mean(), 4), ", sigma =", round(x.std(ddof=0), 4))

In [ ]:
# Verify with scipy and plot the fitted curve for one feature
col = features[0]
x = df[col]
mu, sigma = norm.fit(x)
print("norm.fit ->", mu, sigma)
plt.figure(figsize=(8, 5))
sns.histplot(x, bins=20, stat='density', color='skyblue')
x_vals = np.linspace(x.min(), x.max(), 100)
plt.plot(x_vals, norm.pdf(x_vals, mu, sigma), color='red')
plt.title("Data with MLE Normal Fit: " + col)
plt.show()

# Practical 7

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
df = pd.read_csv('Social_Network_Ads.csv')
X = df[['Age', 'EstimatedSalary']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Choose number of clusters (elbow plot)
wcss = []
for k in range(1, 11):
    wcss.append(KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled).inertia_)
plt.plot(range(1, 11), wcss, marker='o')
plt.xlabel("Number of clusters")
plt.ylabel("WCSS")
plt.title("Elbow Plot")
plt.show()

In [ ]:
# Fit and plot the clusters (k = 3; change to the elbow point)
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)
centers = scaler.inverse_transform(kmeans.cluster_centers_)
plt.figure(figsize=(8, 6))
plt.scatter(df['Age'], df['EstimatedSalary'], c=df['Cluster'], cmap='viridis')
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=200, marker='X')
plt.xlabel("Age")
plt.ylabel("Estimated Salary")
plt.title("Customer Clusters")
plt.show()